# NGB v4 — tuned one-head optimizer comparison

Uses the complete seed intersection shared by SGD, AdamW, and Muon. All uncertainty is run-level. Perplexity intervals are transformed from cross-entropy loss space.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'baseline' / 'ngb']
EXPERIMENT_ROOT = next((p for p in candidates if (p / 'configs' / 'v4_one_head.yaml').is_file()), None)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError('Run from baseline/ngb, baseline/ngb/notebooks, or the repository root')
sys.path.insert(0, str(EXPERIMENT_ROOT / 'src'))

from rg_ngb import (
    SUPPORTED_OPTIMIZERS, diagnostic_summary, discover_matched_seeds,
    final_test_summary, load_config, load_epoch_metrics, load_layer_metrics,
    load_spectral_summary, load_test_results, paired_optimizer_differences,
    plot_epoch_metric, plot_layer_metric, roots, run_diagnostics, run_status_table,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
CONFIG = load_config(EXPERIMENT_ROOT / 'configs' / 'v4_one_head.yaml')
PATHS = roots(CONFIG)
OPTIMIZERS = SUPPORTED_OPTIMIZERS
SEEDS = discover_matched_seeds(PATHS['results'], optimizers=OPTIMIZERS)
PLOT_DIR = PATHS['plots'] / 'comparison'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print('matched completed seeds:', SEEDS)
display(run_status_table(PATHS['results'], optimizers=OPTIMIZERS, seeds=SEEDS))

In [ ]:
epoch_metrics = load_epoch_metrics(PATHS['results'], optimizers=OPTIMIZERS, seeds=SEEDS)
layer_metrics = load_layer_metrics(PATHS['results'], optimizers=OPTIMIZERS, seeds=SEEDS)
spectral_summary = load_spectral_summary(PATHS['results'], optimizers=OPTIMIZERS, seeds=SEEDS)
test_results = load_test_results(PATHS['results'], optimizers=OPTIMIZERS, seeds=SEEDS)
diagnostics = run_diagnostics(PATHS['results'], optimizers=OPTIMIZERS, seeds=SEEDS)
display(epoch_metrics.sort_values(['optimizer', 'seed', 'nominal_epoch']))

## Full and post-transient task trajectories

In [ ]:
for metric in [
    'train_loss', 'val_loss', 'test_loss',
    'train_accuracy', 'val_accuracy', 'test_accuracy',
    'train_perplexity', 'val_perplexity', 'test_perplexity',
    'test_bleu', 'val_generalization_gap', 'test_generalization_gap',
    'weight_norm', 'update_to_weight_ratio', 'grad_norm_pre_clip',
]:
    if metric not in epoch_metrics.columns:
        continue
    plot_epoch_metric(epoch_metrics, metric=metric, optimizers=OPTIMIZERS, output=PLOT_DIR / f'full_{metric}.png')
    plt.show()
    zoom = epoch_metrics[epoch_metrics['nominal_epoch'].ge(0.5)]
    plot_epoch_metric(zoom, metric=metric, optimizers=OPTIMIZERS, title=f'{metric}: epoch 0.5 onward', output=PLOT_DIR / f'zoom_{metric}.png')
    plt.show()

## Optimizer-level and block-resolved WeightWatcher diagnostics

In [ ]:
for metric in ['alpha_median', 'ERG_gap_median', 'num_traps_mean']:
    plot_epoch_metric(spectral_summary, metric=metric, x='epoch', optimizers=OPTIMIZERS, output=PLOT_DIR / f'spectral_{metric}.png')
    if metric == 'alpha_median':
        plt.axhline(2.0, color='black', linestyle='--', linewidth=1.0, label='alpha = 2')
    if metric == 'ERG_gap_median':
        plt.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    plt.show()

for optimizer in OPTIMIZERS:
    for metric in ['alpha', 'ERG_gap', 'num_traps']:
        plot_layer_metric(layer_metrics, optimizer=optimizer, metric=metric, output=PLOT_DIR / f'{optimizer}_layer_{metric}.png')
        plt.show()

## Final, validation-selected, paired, and stability tables

In [ ]:
terminal = final_test_summary(test_results)
paired = paired_optimizer_differences(test_results)
stability = diagnostic_summary(diagnostics)
display(terminal.sort_values(['checkpoint', 'metric', 'optimizer_label']))
display(paired.sort_values(['checkpoint', 'metric', 'contrast']))
display(diagnostics.sort_values(['optimizer', 'seed']))
display(stability.sort_values(['metric', 'optimizer_label']))
terminal.to_csv(PLOT_DIR / 'terminal_summary_95ci.csv', index=False)
paired.to_csv(PLOT_DIR / 'paired_optimizer_differences_95ci.csv', index=False)
diagnostics.to_csv(PLOT_DIR / 'run_diagnostics.csv', index=False)
stability.to_csv(PLOT_DIR / 'diagnostic_summary_95ci.csv', index=False)